In [1]:
import sys
sys.path.append('../../')

### Investigating `minor_ratio` in Waterbirds module

In [ ]:
from smoothAttributionPrior.module.utils.spurious_dataset import SpuriousDataset
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

root = '/home/data/SpuriousCatDogVer2/img'

transform_ = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
)

dataset_train = SpuriousDataset(root=os.path.join(root, "train"), transform=transform_)
dataset_minor_train = SpuriousDataset(root=os.path.join(root, "minor_train"), transform=transform_)

# self.dataset_train = self.combine_dataset(
#     self.dataset_train, dataset_minor_train, self.hparams.minor_ratio
# )

dataset_major_val = SpuriousDataset(root=os.path.join(root, "major_val"), transform=transform_)
dataset_minor_val = SpuriousDataset(root=os.path.join(root, "minor_val"), transform=transform_)

dataset_major_test = SpuriousDataset(root=os.path.join(root, "test"), transform=transform_)
dataset_minor_test = SpuriousDataset(root=os.path.join(root, "corrupted_test"), transform=transform_)


for dset in [dataset_train, dataset_minor_train, dataset_major_val, dataset_major_val, dataset_major_test, dataset_minor_test]:
    gs = []
    for _, y, g in DataLoader(dset, batch_size=128, num_workers=16):
        gs.append(g)

    g = torch.cat(gs)
    arr = g[:,:2]
    print(arr.shape)
    print(sum(arr[:,0] == arr[:,1]) / len(arr), len(arr))

print('Works well')


# SpuriousDataModule

In [ ]:
from smoothAttributionPrior.module.utils.data_module import SpuriousDataModule
dm = SpuriousDataModule(data_dir='/home/data', dataset='SpuriousCatDogVer2', minor_ratio=0)

In [ ]:
dm.prepare_data()
dm.setup()

In [ ]:
tr_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()
test_loader = dm.test_dataloader()

In [ ]:
for loader in [tr_loader] + val_loader + test_loader:
    leng = 0
    for data in loader:
        x, y, g = data
        
        leng += len(y)
    print(leng)

# SpuriousConceptDataModule

In [ ]:
import sys
sys.path.append('../')
from smoothAttributionPrior.module.utils.data_module import SpuriousConceptDataModule

root = '/home/data/'
dm = SpuriousConceptDataModule(spurious_root=root, concept_root=root, dataset='SpuriousCatDogVer2', batch_size_train=8, batch_size_test=8, minor_ratio=0.5)
dm.prepare_data()
dm.setup()


In [ ]:
tr_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()
test_loader = dm.test_dataloader()

In [ ]:
for loader in list(tr_loader.values()) + val_loader + test_loader:
    leng = 0
    for data in loader:
        try:
            x, y, g = data
        except:
            x, y = data
        leng += len(y)
    print(leng)

In [ ]:
import matplotlib.pyplot as plt
import torch 
mean = torch.tensor((0.485, 0.456, 0.406)).reshape(1,1,3)
std = torch.tensor((0.229, 0.224, 0.225)).reshape(1,1,3)
def ten2im(dat):
    return dat.permute(1,2,0) * std + mean
    
fig, axes = plt.subplots(3,8,figsize=(16,6))
for i, data in enumerate(tr_loader['classifier']):
    for j in range(8):
        axes[i,j].imshow(ten2im(data[0][j]))
        axes[i,j].axis('off')
    if i == 2:
        break
